# Multi-Source OOD Computation

This notebook computes and caches NPE logML/PMP estimates, summary diagnostics, and posterior diagnostics for all registered observed datasets.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_DIR = Path("/Users/yimingzang/Documents/Project/benchmark2")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from benchmark.examples.diffusion.config import TrainingConfig
from benchmark.examples.diffusion.results.multisource_pipeline import (
    all_observed_paths,
    compute_or_load_all_observed,
)
from benchmark.examples.diffusion.results.observed_datasets import OBSERVED_DATASETS


INFO:jax._src.xla_bridge:Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/anaconda3/envs/benchmark2/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file), '/usr/local/lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache)
INFO:bayesflow:Using backend 'jax'


In [ ]:
from tqdm.auto import tqdm as original_tqdm
import bayesflow.approximators.helpers.samplers as bf_samplers
import bayesflow.approximators.helpers.conditions as bf_conditions


def quiet_tqdm(*args, **kwargs):
    kwargs["disable"] = True
    return original_tqdm(*args, **kwargs)


bf_samplers.tqdm = quiet_tqdm
bf_conditions.tqdm = quiet_tqdm


## Configuration


In [ ]:
summary_configs = {
    "S=D": TrainingConfig(summary_multiplier=1),
    "S=2D": TrainingConfig(summary_multiplier=2),
    "S=4D": TrainingConfig(summary_multiplier=4),
    "S=6D": TrainingConfig(summary_multiplier=6),
}

metric = "l2"
num_samples = 2048
mmd_samples = 512
batch_size = 8
recompute = False
recompute_references = False

OBSERVED_DATASETS


('empirical',
 'simulated_from_m0',
 'simulated_from_m1',
 'simulated_from_m2',
 'simulated_from_m3',
 'm3_fast_30',
 'm3_slow_30',
 'm3_fast_slow_30')

## Compute Or Load Cached Results


In [ ]:
diagnostics_by_summary = {}

for label, config in summary_configs.items():
    print(f"Computing/loading {label} ({config.summary_label})")
    diagnostics_by_summary[label] = compute_or_load_all_observed(
        config=config,
        metric=metric,
        num_samples=num_samples,
        mmd_samples=mmd_samples,
        batch_size=batch_size,
        recompute=recompute,
        recompute_references=recompute_references,
    )


Computing/loading S=D (S1D)
Computing/loading S=2D (S2D)
Computing/loading S=4D (S4D)
Computing/loading S=6D (S6D)


## Cached Files


In [ ]:
cache_rows = []
for label, config in summary_configs.items():
    for name, path in all_observed_paths(config.summary_label, metric=metric).items():
        cache_rows.append(
            {
                "summary": label,
                "file_type": name,
                "path": str(path),
                "exists": path.exists(),
            }
        )

cache_files = pd.DataFrame(cache_rows)
cache_files


,summary,file_type,path,exists
0,S=D,results,/Users/yimingzang/Documents/Project/benchmark2...,True
1,S=D,diagnostic,/Users/yimingzang/Documents/Project/benchmark2...,True
2,S=D,posterior,/Users/yimingzang/Documents/Project/benchmark2...,True
3,S=D,posterior_plot,/Users/yimingzang/Documents/Project/benchmark2...,True
4,S=2D,results,/Users/yimingzang/Documents/Project/benchmark2...,True
5,S=2D,diagnostic,/Users/yimingzang/Documents/Project/benchmark2...,True
6,S=2D,posterior,/Users/yimingzang/Documents/Project/benchmark2...,True
7,S=2D,posterior_plot,/Users/yimingzang/Documents/Project/benchmark2...,True
8,S=4D,results,/Users/yimingzang/Documents/Project/benchmark2...,True
9,S=4D,diagnostic,/Users/yimingzang/Documents/Project/benchmark2...,True


## Sanity Checks


In [ ]:
coverage_rows = []
for label, frames in diagnostics_by_summary.items():
    for name, frame in frames.items():
        coverage_rows.append(
            {
                "summary": label,
                "frame": name,
                "rows": len(frame),
                "datasets": ", ".join(frame["dataset"].drop_duplicates()),
                "n_datasets": frame["dataset"].nunique(),
            }
        )

coverage = pd.DataFrame(coverage_rows)
coverage


,summary,frame,rows,datasets,n_datasets
0,S=D,results,136,"empirical, simulated_from_m0, simulated_from_m...",8
1,S=D,diagnostic,136,"empirical, simulated_from_m0, simulated_from_m...",8
2,S=D,posterior,544,"empirical, simulated_from_m0, simulated_from_m...",8
3,S=D,posterior_plot,544,"empirical, simulated_from_m0, simulated_from_m...",8
4,S=2D,results,136,"empirical, simulated_from_m0, simulated_from_m...",8
5,S=2D,diagnostic,136,"empirical, simulated_from_m0, simulated_from_m...",8
6,S=2D,posterior,544,"empirical, simulated_from_m0, simulated_from_m...",8
7,S=2D,posterior_plot,544,"empirical, simulated_from_m0, simulated_from_m...",8
8,S=4D,results,136,"empirical, simulated_from_m0, simulated_from_m...",8
9,S=4D,diagnostic,136,"empirical, simulated_from_m0, simulated_from_m...",8


In [ ]:
posterior_summary = []
for label, frames in diagnostics_by_summary.items():
    frame = frames["posterior"]
    posterior_summary.append(
        frame.groupby(["dataset", "model"], sort=False)
        .agg(
            mean_mmd=("posterior_mmd", "mean"),
            median_mmd=("posterior_mmd", "median"),
            mean_posterior_mean_rmse=("posterior_mean_rmse", "mean"),
        )
        .assign(summary_dimension=label)
        .reset_index()
    )

posterior_summary = pd.concat(posterior_summary, ignore_index=True)
posterior_summary


,dataset,model,mean_mmd,median_mmd,mean_posterior_mean_rmse,summary_dimension
0,empirical,m0,0.219341,0.154009,0.238050,S=D
1,empirical,m1,0.227051,0.175034,0.194805,S=D
2,empirical,m2,0.211853,0.221962,0.220987,S=D
3,empirical,m3,0.306619,0.212100,0.259090,S=D
4,simulated_from_m0,m0,0.010768,0.008996,0.040133,S=D
...,...,...,...,...,...,...
123,m3_slow_30,m3,0.065447,0.056301,0.102677,S=6D
124,m3_fast_slow_30,m0,0.281155,0.179708,0.416993,S=6D
125,m3_fast_slow_30,m1,0.442371,0.418471,0.477028,S=6D
126,m3_fast_slow_30,m2,0.095229,0.064474,0.140913,S=6D
